In [33]:
# Plot the uncorrected in-plane deformation time-series from the Messung24 dataset

%reset -f
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io

plt.close("all")
plt.style.use('default')

START_FRAME = 0
END_FRAME = 10000

# Open file dialog to select a .mat file
pitch_angle_file = "Messung24/pitch_bladeA_messung24.mat"
file_path_wo_pitch_corr = "Messung24/Blade_0_AOI_2_[3939. 1147.]_woPitchCorr.csv"

pitch_angle_data = scipy.io.loadmat(pitch_angle_file)
pitch_angle_data = pitch_angle_data["pitch_wea_bladeA"].flatten()
pitch_angle_data = pitch_angle_data - pitch_angle_data[0]

df_w_pitch = pd.read_csv(file_path_wo_pitch_corr, sep=";", decimal=',', skiprows=1, header=0)
y_pos = df_w_pitch["Y [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
sigma = df_w_pitch["sigma [pixel]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
deformation_V = df_w_pitch["V [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]

bad_sigmas = np.nonzero(sigma < 0)[0]
deformation_V[bad_sigmas] = 0

fig, (ax, ax1) = plt.subplots(2, 1, sharex=True)

ax.plot(deformation_V)
ax.set_ylabel("In-Plane Deformation [mm]")
ax.grid(True)
ax.set_xlim(START_FRAME, END_FRAME)
plt.tight_layout()

ax1.plot(pitch_angle_data)
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Pitch Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
fig.align_ylabels()

plt.savefig('../../graphics/problem_statement.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

In [34]:
# Plot the uncorrected and corrected in-plane deformation time-series from the Messung24 dataset

%reset -f
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.io

# Plot deformation

plt.close("all")
plt.style.use('default')

START_FRAME = 0
END_FRAME = 10000

# Open file dialog to select a .mat file
pitch_angle_file = "Messung24/pitch_bladeA_messung24.mat"

file_path_wo_pitch_corr = "Messung24/Blade_0_AOI_2_[3939. 1147.]_woPitchCorr.csv"
file_path_w_pitch_corr = "Messung24/Blade_0_AOI_2_[3939. 1147.]_wPitchCorr.csv"
file_list = [file_path_wo_pitch_corr, file_path_w_pitch_corr]

pitch_angle_data = scipy.io.loadmat(pitch_angle_file)
pitch_angle_data = pitch_angle_data["pitch_wea_bladeA"].flatten()
pitch_angle_data = pitch_angle_data - pitch_angle_data[0]

fig, (ax, ax1) = plt.subplots(2, 1, sharex=True)
for file in file_list:
    df_w_pitch = pd.read_csv(file, sep=";", decimal=',', skiprows=1, header=0)
    y_pos = df_w_pitch["Y [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    sigma = df_w_pitch["sigma [pixel]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    deformation_V = df_w_pitch["V [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]

    bad_sigmas = np.nonzero(sigma < 0)[0]
    deformation_V[bad_sigmas] = 0

    ax.plot(deformation_V)

ax.set_ylabel("In-Plane Deformation [mm]")
ax.grid(True)
ax.set_xlim(START_FRAME, END_FRAME)
ax.legend(["w/o Pitch Correction", "w/ Pitch Correction"])
plt.tight_layout()

ax1.plot(pitch_angle_data)
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Pitch Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
fig.align_ylabels()

plt.savefig('../../graphics/results_messung24.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

In [42]:
# Plot the results from the Blender_6k synthetic dataset

%reset -f
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import medfilt

# Plot deformation

plt.close("all")
plt.style.use('default')
PLOT_SCALE_FACTOR = 1.5

START_FRAME = 0
END_FRAME = 6000

pitch_angle_file = "Blender_6k/blender_inputs/pitch_angle_data_6k.csv"

file_path_wo_pitch_corr = "Blender_6k/SchlagSchwenk_woPitchCorr/Blade_0_AOI_2_[4125. 1421.].csv"
file_path_w_pitch_corr = "Blender_6k/SchlagSchwenk_wPitchCorr_1/Blade_0_AOI_2_[4125. 1421.].csv"
file_list = [file_path_wo_pitch_corr, file_path_w_pitch_corr]

blender_baseline = np.loadtxt("Blender_6k/blender_inputs/deformation_V_6k.csv")

pitch_angle_data = np.loadtxt(pitch_angle_file)
pitch_angle_data = pitch_angle_data - pitch_angle_data[0]

fig, (ax, ax2, ax1) = plt.subplots(3, 1, sharex=True)
ax.plot(blender_baseline, color="tab:green")

for file in file_list:
    df_w_pitch = pd.read_csv(file, sep=";", decimal=',', skiprows=1, header=0)
    y_pos = df_w_pitch["Y [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    sigma = df_w_pitch["sigma [pixel]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    deformation_V = df_w_pitch["V [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    deformation_V = medfilt(-deformation_V)

    bad_sigmas = np.nonzero(sigma < 0)[0]
    deformation_V[bad_sigmas] = 0

    ax.plot(deformation_V)

    absolute_error = deformation_V - blender_baseline

    ax2.plot(absolute_error)

ax.set_ylabel("In-Plane Deformation [mm]")
ax.grid(True)
ax.set_xlim(START_FRAME, END_FRAME)
ax.legend(["Blender Baseline", "w/o Pitch Correction", "w/ Pitch Correction"], loc="lower left")

ax2.grid(True)
ax2.set_ylabel("Absolute Error [mm]")

ax1.plot(pitch_angle_data[START_FRAME:END_FRAME])
ax1.set_xlabel("Frame No.")
ax1.set_ylabel("Pitch Angle Offset [°]")
ax1.grid(True)
ax1.set_xlim(START_FRAME, END_FRAME)
fig.align_ylabels()
plt.tight_layout()

fig.set_size_inches(PLOT_SCALE_FACTOR * np.array((6.4, 4.8)))
plt.savefig('../../graphics/results_blender.pdf', format='pdf', transparent=True, bbox_inches='tight')
plt.show()

In [47]:
# Plot error bars

%reset -f
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import medfilt

# Plot deformation

plt.close("all")
plt.style.use('default')
PLOT_SCALE_FACTOR = 1

START_FRAME = 0
END_FRAME = 6000

pitch_angle_file = "Blender_6k/blender_inputs/pitch_angle_data_6k.csv"

file_path_wo_pitch_corr = "Blender_6k/SchlagSchwenk_woPitchCorr/Blade_0_AOI_2_[4125. 1421.].csv"
file_path_w_pitch_corr = "Blender_6k/SchlagSchwenk_wPitchCorr_1/Blade_0_AOI_2_[4125. 1421.].csv"
file_list = [file_path_wo_pitch_corr, file_path_w_pitch_corr]

blender_baseline = np.loadtxt("Blender_6k/blender_inputs/deformation_V_6k.csv")

pitch_angle_data = np.loadtxt(pitch_angle_file)
pitch_angle_data = pitch_angle_data - pitch_angle_data[0]

bin_centers = np.arange(np.round(np.min(blender_baseline), decimals=-1),
                        np.round(np.max(blender_baseline), decimals=-1) + 1,
                        np.round((np.round(np.max(blender_baseline), decimals=-1) - np.round(np.min(blender_baseline),
                                                                                             decimals=-1)) / 10, 0))

bin_width = np.round(
    (np.round(np.max(blender_baseline), decimals=-1) - np.round(np.min(blender_baseline), decimals=-1)) / 10,
    -1)
bar_width = 25

fig1, ax3 = plt.subplots(1, 1, figsize=(6, 5))

for i, file in enumerate(file_list):
    df_w_pitch = pd.read_csv(file, sep=";", decimal=',', skiprows=1, header=0)
    y_pos = df_w_pitch["Y [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    sigma = df_w_pitch["sigma [pixel]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    deformation_V = df_w_pitch["V [mm]"].to_numpy(copy=True)[START_FRAME:END_FRAME]
    deformation_V = medfilt(-deformation_V)

    bad_sigmas = np.nonzero(sigma < 0)[0]
    deformation_V[bad_sigmas] = 0

    absolute_error = deformation_V - blender_baseline

    binned_means = []
    binned_errors = []

    for center in bin_centers:
        mask = (deformation_V >= center - bin_width / 2) & \
               (deformation_V < center + bin_width / 2)
        if np.any(mask):
            subset_errors = absolute_error[mask]
            binned_means.append(np.mean(subset_errors))

            n_points = len(subset_errors)
            sem = np.std(subset_errors, ddof=1) / np.sqrt(n_points) if n_points > 1 else 0.0
            binned_errors.append(sem)

    offset = (i - 0.5) * bar_width
    label_text = "w/o Pitch Correction" if i == 0 else "w/ Pitch Correction"

    ax3.bar(
        bin_centers + offset,
        binned_means,
        width=bar_width,
        linewidth=0.8,
        edgecolor="black",
        yerr=binned_errors,
        capsize=3,
        error_kw={'elinewidth': 1, 'ecolor': 'black'},
        label=label_text
    )

    MAE = np.abs(absolute_error).mean()
    AE_max = np.abs(absolute_error).max()
    SEM = np.abs(absolute_error).std() / np.sqrt(n_points)

    print(f"{label_text}")
    print(f"MAE= {MAE} mm")
    print(f"MAximum = {AE_max} mm")
    print(f"SEM = {SEM} mm")
    print("----------------------------------------------------------")

ax3.axhline(0, color='black', linewidth=0.8)
ax3.set_xlabel("In-Plane Deformation [mm]")
ax3.set_ylabel("Mean Absolute Error [mm]")
ax3.set_xticks(bin_centers)
ax3.grid(True, linestyle='-', alpha=0.5, color='gray')
ax3.legend()
fig1.tight_layout()
fig1.set_size_inches(PLOT_SCALE_FACTOR * np.array((6.4, 4.8)))
fig1.savefig('../../graphics/error_bars_binned.pdf', format='pdf', bbox_inches='tight')

plt.show()

w/o Pitch Correction
MAE= 91.15811914816902 mm
MAximum = 353.00305603515625 mm
SEM = 3.280247633126412 mm
----------------------------------------------------------
w/ Pitch Correction
MAE= 43.59800302641576 mm
MAximum = 134.3902436328125 mm
SEM = 1.919434292009913 mm
----------------------------------------------------------


In [49]:
# Plot the pitch correction process

%reset -f
%matplotlib qt

import matplotlib.pyplot as plt
import numpy as np

from coordsysalign.multiprocessing_fns import read_file
from coordsysalign.transformation_fns import calculate_circle_rotation_matrix_2, rotation_matrix_z, \
    find_blade_axis


def generate_cylinder_mesh(p0, axis_dir, radius, height, num_segments=300):
    axis_dir = axis_dir / np.linalg.norm(axis_dir)

    # Compute perpendicular basis vectors u and v
    if np.allclose(axis_dir[:2], 0):
        u = np.array([1.0, 0.0, 0.0])
    else:
        u = np.array([-axis_dir[1], axis_dir[0], 0.0])
    u = u / np.linalg.norm(u)
    v = np.cross(axis_dir, u)

    theta = np.linspace(0, 2 * np.pi, num_segments)
    h = np.linspace(-height / 2, height / 2, 2)
    THETA, H = np.meshgrid(theta, h)

    X = (
            p0[0]
            + H * axis_dir[0]
            + radius * (np.cos(THETA) * u[0] + np.sin(THETA) * v[0])
    )
    Y = (
            p0[1]
            + H * axis_dir[1]
            + radius * (np.cos(THETA) * u[1] + np.sin(THETA) * v[1])
    )
    Z = (
            p0[2]
            + H * axis_dir[2]
            + radius * (np.cos(THETA) * u[2] + np.sin(THETA) * v[2])
    )

    return X, Y, Z


plt.close("all")
plt.style.use('default')

blade_name_list = ["A", "B", "C"]
hub_aoi_array = np.array([0, 1, 2])
aoi_blade_num_array = np.array([0.0, 1.0, 2.0, 1.0, 1.0, 1.0, 2.0, 2.0, 2.0, 0.0, 0.0, 0.0])

out_file = "Messung24/messung24_000982_0.out"

visibility_f0, coordinates_f0, xyz_sigmas_f0, aoi_number_f0, index_in_aoi_f0 = read_file(out_file, None)
visible_idxs_f0 = np.nonzero(visibility_f0 > 0)[::10]
coordinates_f0 = coordinates_f0[visible_idxs_f0]
aoi_number_f0 = aoi_number_f0[visible_idxs_f0]

LEGEND_FLAG = True

fig = plt.figure()
ax = fig.add_subplot(231, projection="3d")
ax2 = fig.add_subplot(232, projection="3d")
ax3 = fig.add_subplot(233, projection="3d")
ax4 = fig.add_subplot(234, projection="3d")
ax5 = fig.add_subplot(235, projection="3d")
ax6 = fig.add_subplot(236, projection="3d")

p0_list = []
axis_dir_list = []
blade_rad_list = []
for blade_idx, blade_name in enumerate(blade_name_list[:1]):

    # Find the indices/AoIs in the current blade
    aoi_in_blade_array = np.nonzero(aoi_blade_num_array == blade_idx)[0]

    for aoi_in_blade in aoi_in_blade_array[1::-1]:
        # tip_aoi = aoi_in_blade_array[np.argmax(aoi_to_blade_aoi[aoi_in_blade_array])]
        tip_indices = np.nonzero(aoi_number_f0 == aoi_in_blade)[0]

        if tip_indices.size != 0:
            tip_points = coordinates_f0[tip_indices]
            tip_avg_coord = np.mean(tip_points, axis=0)
            break
        else:
            continue

    # Isolate the points belonging to the AoIs closest to the rotor hub
    blade_hub_idx_array = np.nonzero(aoi_number_f0 == hub_aoi_array[blade_idx])[0]
    hub_points = coordinates_f0[blade_hub_idx_array]
    hub_avg_coord = np.mean(hub_points, axis=0)

    blade_anchor, blade_dir, blade_rad = find_blade_axis(hub_points, tip_points)

    # # Indices for the blade
    complete_blade_idx_array = np.nonzero(np.isin(aoi_number_f0, aoi_in_blade_array))[0]
    blade_points = coordinates_f0[complete_blade_idx_array]

    height = 1500  #h_max - h_min

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        blade_anchor, blade_dir, blade_rad, height=height
    )

    # Cylinder central axis line
    axis_pts = np.array(
        [
            blade_anchor - blade_dir * (height / 2 + 50),
            blade_anchor + blade_dir * (height / 2 + 50),
        ]
    )

    # Scatter plot of original points
    ax.scatter(
        hub_points[:, 0],
        hub_points[:, 1],
        hub_points[:, 2],
        color="blue",
        s=10,
        label="Points used for Cylinder fit"
    )

    ax.scatter(
        blade_points[:, 0],
        blade_points[:, 1],
        blade_points[:, 2],
        color="yellow",
        s=10,
        alpha=0.4,
        label="Raw Blade Points"
    )

    # ax.plot(
    #     axis_pts[:, 0],
    #     axis_pts[:, 1],
    #     axis_pts[:, 2],
    #     "b--",
    #     linewidth=2,
    #     label="Cylinder Axis",
    # )
    ax.plot(
        blade_anchor[0],
        blade_anchor[1],
        blade_anchor[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(blade_anchor, 1)} mm",
    )

    ax.quiver(
        blade_anchor[0],
        blade_anchor[1],
        blade_anchor[2],
        blade_dir[0],
        blade_dir[1],
        blade_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(blade_dir, 3)}",
        length=10000,
    )

    # Cylinder wireframe mesh
    ax.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
        label="Fitted Cylinder"
    )

    centered_points = hub_points - blade_anchor
    centered_blade_points = blade_points - blade_anchor
    centered_center_p0 = blade_anchor - blade_anchor

    # Scatter plot of original points

    ax2.scatter(
        centered_blade_points[:, 0],
        centered_blade_points[:, 1],
        centered_blade_points[:, 2],

        color="yellow",
        s=10,
        alpha=0.4,
    )

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        centered_center_p0, blade_dir, blade_rad, height=height
    )

    ax2.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
    )

    ax2.plot(
        centered_center_p0[0],
        centered_center_p0[1],
        centered_center_p0[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(centered_center_p0, 1)} mm",
    )

    ax2.quiver(
        centered_center_p0[0],
        centered_center_p0[1],
        centered_center_p0[2],
        blade_dir[0],
        blade_dir[1],
        blade_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(blade_dir, 3)}",
        length=10000,
    )

    first_rot = calculate_circle_rotation_matrix_2(blade_dir, 1)

    rotated_points = np.dot(first_rot, centered_points.T).T
    rotated_blade_points = np.dot(first_rot, centered_blade_points.T).T
    rotated_center_p0 = np.dot(first_rot, centered_center_p0.T).T

    rotated_axis_dir = np.matmul(first_rot, blade_dir)

    # Scatter plot of original points

    ax3.scatter(
        rotated_blade_points[:, 0],
        rotated_blade_points[:, 1],
        rotated_blade_points[:, 2],

        color="yellow",
        s=10,
        alpha=0.4,
    )

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        rotated_center_p0, rotated_axis_dir, blade_rad, height=height
    )

    ax3.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
    )

    ax3.plot(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(rotated_center_p0, 1)} mm",
    )

    ax3.quiver(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        rotated_axis_dir[0],
        rotated_axis_dir[1],
        rotated_axis_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(rotated_axis_dir, 3)}",
        length=10000,
    )

    second_rot = rotation_matrix_z(-np.pi / 4)

    old_rotated_blade_points = rotated_blade_points

    rotated_points = np.dot(second_rot, rotated_points.T).T
    rotated_blade_points = np.dot(second_rot, rotated_blade_points.T).T
    rotated_center_p0 = np.dot(second_rot, rotated_center_p0.T).T

    rotated_axis_dir = np.matmul(second_rot, rotated_axis_dir)

    ax4.scatter(
        old_rotated_blade_points[:, 0],
        old_rotated_blade_points[:, 1],
        old_rotated_blade_points[:, 2],

        color="yellow",
        s=10,
        alpha=0.4,
        label="w/ Pitch Angle"
    )

    ax4.scatter(
        rotated_blade_points[:, 0],
        rotated_blade_points[:, 1],
        rotated_blade_points[:, 2],

        color="lime",
        s=10,
        alpha=0.4,
        label="w/o Pitch Angle"
    )

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        rotated_center_p0, rotated_axis_dir, blade_rad, height=height
    )

    ax4.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
    )

    ax4.plot(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(rotated_center_p0, 1)} mm",
    )

    ax4.quiver(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        rotated_axis_dir[0],
        rotated_axis_dir[1],
        rotated_axis_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(rotated_axis_dir, 3)}",
        length=10000,
    )

    third_rot = np.linalg.inv(first_rot)

    rotated_points = np.dot(third_rot, rotated_points.T).T
    rotated_blade_points = np.dot(third_rot, rotated_blade_points.T).T
    rotated_center_p0 = np.dot(third_rot, rotated_center_p0.T).T

    rotated_axis_dir = np.matmul(third_rot, rotated_axis_dir)

    ax5.scatter(
        centered_blade_points[:, 0],
        centered_blade_points[:, 1],
        centered_blade_points[:, 2],

        color="yellow",
        s=10,
        alpha=0.4,
    )

    ax5.scatter(
        rotated_blade_points[:, 0],
        rotated_blade_points[:, 1],
        rotated_blade_points[:, 2],

        color="lime",
        s=10,
        alpha=0.4,
    )

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        rotated_center_p0, rotated_axis_dir, blade_rad, height=height
    )

    ax5.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
    )

    ax5.plot(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(rotated_center_p0, 1)} mm",
    )

    ax5.quiver(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        rotated_axis_dir[0],
        rotated_axis_dir[1],
        rotated_axis_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(rotated_axis_dir, 3)}",
        length=10000,
    )

    rotated_points = rotated_points + blade_anchor
    rotated_blade_points = rotated_blade_points + blade_anchor
    rotated_center_p0 = rotated_center_p0 + blade_anchor

    # Scatter plot of original points
    ax6.scatter(
        rotated_blade_points[:, 0],
        rotated_blade_points[:, 1],
        rotated_blade_points[:, 2],
        color="lime",
        s=10,
    )

    ax6.scatter(
        blade_points[:, 0],
        blade_points[:, 1],
        blade_points[:, 2],

        color="yellow",
        s=10,
        alpha=0.4,
    )

    X_mesh, Y_mesh, Z_mesh = generate_cylinder_mesh(
        rotated_center_p0, rotated_axis_dir, blade_rad, height=height
    )

    ax6.plot_wireframe(
        X_mesh,
        Y_mesh,
        Z_mesh,
        color="cyan",
        alpha=0.4,
    )

    ax6.plot(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        "x",
        linewidth=2,
        color="magenta",
        label=f"Cylinder Origin: {np.round(rotated_center_p0, 1)} mm",
    )

    ax6.quiver(
        rotated_center_p0[0],
        rotated_center_p0[1],
        rotated_center_p0[2],
        rotated_axis_dir[0],
        rotated_axis_dir[1],
        rotated_axis_dir[2],
        normalize=True,
        color="magenta",
        label=f"Blade axis: {np.round(rotated_axis_dir, 3)}",
        length=10000,
    )

p0_array = np.array(p0_list)
axis_dir_array = np.array(axis_dir_list)
blade_rad_array = np.array(blade_rad_list)

# Labels, title, and display settings
ax.set_title("1. Read points from .out file")
ax.set_xlabel("x", weight='bold')
ax.set_ylabel("y", weight='bold')
ax.set_zlabel("z", weight='bold')
ax.axis("equal")

ax2.set_title("2. Bring the blade origin to the global origin")
ax2.set_xlabel("x", weight='bold')
ax2.set_ylabel("y", weight='bold')
ax2.set_zlabel("z", weight='bold')
ax2.axis("equal")

ax3.set_title("3. Align blade axis to the global z-axis")
ax3.set_xlabel("x", weight='bold')
ax3.set_ylabel("y", weight='bold')
ax3.set_zlabel("z", weight='bold')
ax3.axis("equal")

ax4.set_title("4. Rotate around the z-axis to eliminate the current blade pitch")
ax4.set_xlabel("x", weight='bold')
ax4.set_ylabel("y", weight='bold')
ax4.set_zlabel("z", weight='bold')
ax4.axis("equal")

ax5.set_title("5. Return the blade axis to its original orientation")
ax5.set_xlabel("x", weight='bold')
ax5.set_ylabel("y", weight='bold')
ax5.set_zlabel("z", weight='bold')
ax5.axis("equal")

ax6.set_title("6. Return the blade origin to its original position")
ax6.set_xlabel("x", weight='bold')
ax6.set_ylabel("y", weight='bold')
ax6.set_zlabel("z", weight='bold')
ax6.axis("equal")

if LEGEND_FLAG:
    ax.legend()
    ax2.legend()
    ax3.legend()
    ax4.legend()
    ax5.legend()
    ax6.legend()

#fig.set_size_inches(25, 10)
#plt.savefig('../docs/graphics/ansatz.pdf', format='pdf', transparent=True, bbox_inches='tight')
